In [1]:
import mxnet as mx

data = mx.symbol.Variable('data')
# 3*224*224 => 96*54*54
conv1 = mx.sym.Convolution(name='conv1', data=data, kernel=(11, 11), stride=(4, 4), num_filter=96)
relu1 = mx.sym.Activation(data=conv1, act_type="relu")
# 96*54*54 => 96*26*26
pool1 = mx.sym.Pooling(data=relu1, pool_type="max", kernel=(3, 3), stride=(2,2))
# 96*26*26 => 256*26*26
conv2 = mx.sym.Convolution(name='conv2', data=pool1, kernel=(5, 5), pad=(2, 2), num_filter=256)
relu2 = mx.sym.Activation(data=conv2, act_type="relu")
# 256*26*26 => 256*12*12
pool2 = mx.sym.Pooling(data=relu2, kernel=(3, 3), stride=(2, 2), pool_type="max")
# 256*12*12 => 384*12*12
conv3 = mx.sym.Convolution(name='conv3', data=pool2, kernel=(3, 3), pad=(1, 1), num_filter=384)
relu3 = mx.sym.Activation(data=conv3, act_type="relu")
# 384*12*12 => 384*12*12
conv4 = mx.sym.Convolution(name='conv4',data=relu3, kernel=(3, 3), pad=(1, 1), num_filter=384)
relu4 = mx.sym.Activation(data=conv4, act_type="relu")
# 384*12*12 => 256*12*12
conv5 = mx.sym.Convolution(name='conv5', data=relu4, kernel=(3, 3), pad=(1, 1), num_filter=256)
relu5 = mx.sym.Activation(data=conv5, act_type="relu")
# 256*12*12 => 256*5*5
pool3 = mx.sym.Pooling(data=relu5, kernel=(3, 3), stride=(2, 2), pool_type="max")
# 256*5*5 => 6400
flatten = mx.sym.Flatten(data=pool3)
# 6400 => 4096
fc1 = mx.sym.FullyConnected(name='fc1', data=flatten, num_hidden=4096)
relu6 = mx.sym.Activation(data=fc1, act_type="relu")
dropout1 = mx.sym.Dropout(data=relu6, p=0.5)
# 4096 => 4096
fc2 = mx.sym.FullyConnected(name='fc2', data=dropout1, num_hidden=4096)
relu7 = mx.sym.Activation(data=fc2, act_type="relu")
dropout2 = mx.sym.Dropout(data=relu7, p=0.5)
# 4096 => 1000
fc3 = mx.sym.FullyConnected(name='fc3', data=dropout2, num_hidden=1000)
softmax = mx.sym.SoftmaxOutput(data=fc3, name='softmax')

shape = {"data" : (32, 3, 224, 224)}
mx.viz.print_summary(symbol=softmax, shape=shape)

C:\ProgramData\Miniconda3\lib\site-packages\h5py\__init__.py:72: UserWarning: h5py is running against HDF5 1.10.2 when it was built against 1.10.3, this may cause problems
  '{0}.{1}.{2}'.format(*version.hdf5_built_version_tuple)


________________________________________________________________________________________________________________________
Layer (type)                                        Output Shape            Param #     Previous Layer                  
data(null)                                          3x224x224               0                                           
________________________________________________________________________________________________________________________
conv1(Convolution)                                  96x54x54                34944       data                            
________________________________________________________________________________________________________________________
activation0(Activation)                             96x54x54                0           conv1                           
________________________________________________________________________________________________________________________
pooling0(Pooling)               

In [5]:
#-*-coding:utf-8-*-
#import sys
#reload(sys)
#sys.setdefaultencoding('utf-8') # 进一步要求 python 使用 UTF-8

import numpy as np
import os
import gzip
import struct
import logging
import mxnet as mx
import matplotlib.pyplot as plt # 这是常用的绘图库

logging.getLogger().setLevel(logging.DEBUG)

def read_data(label_url, image_url): # 读入训练数据
    with gzip.open(label_url) as flbl: # 打开标签文件
        magic, num = struct.unpack(">II", flbl.read(8)) # 读入标签文件头
        label = np.fromstring(flbl.read(), dtype=np.int8) # 读入标签内容
    with gzip.open(image_url, 'rb') as fimg: # 打开图像文件
        magic, num, rows, cols = struct.unpack(">IIII", fimg.read(16)) # 读入图像文件头
        image = np.fromstring(fimg.read(), dtype=np.uint8) # 读入图像内容
        image = image.reshape(len(label), 1, rows, cols) # 设置为正确的数组格式
        image = image.astype(np.float32)/255.0 # 归一化到0到1区间
    return (label, image)

# 读入数据
(train_lbl, train_img) = read_data('train-labels-idx1-ubyte.gz', 'train-images-idx3-ubyte.gz')
(val_lbl, val_img) = read_data('t10k-labels-idx1-ubyte.gz', 't10k-images-idx3-ubyte.gz')

batch_size = 32 # 批大小

# 用于辅助定义网络，这是深度卷积网络中常用的"卷积-BN-非线性"模块
def CBA(src, suffix, num_filter, kernel, pad):
    conv = mx.sym.Convolution(data=src, name="conv"+suffix, kernel=(kernel,kernel), pad=(pad,pad), num_filter=num_filter)
    bn = mx.sym.BatchNorm(data=conv, name="bn"+suffix, fix_gamma=False)
    act = mx.sym.Activation(data=bn, name="act"+suffix, act_type="relu")
    return act

# 由于辅助定义网络，这里在每2个卷积层后加1个池化层
def LAYER(src, layer, num_filter, pad):
    conv1 = CBA(src, layer+"1", num_filter, 3, pad)
    conv2 = CBA(conv1, layer+"2", num_filter, 3, pad)
    pool = mx.sym.Pooling(data=conv2, name="pool"+layer, pool_type="max", kernel=(2,2), stride=(2,2))
    return pool

# 设置网络
data = mx.symbol.Variable('data')
# 将3*28*28变换为32*14*14
net = LAYER(data, "1", 32, 1)
# 将32*14*14变换为64*7*7
net = LAYER(net, "2", 64, 1)
# 将64*7*7变换为64*3*3
net = LAYER(net, "3", 64, 1)
# 将64*3*3变换为128*1*1
net = CBA(net, "4", 128, 3, 0)
# 将128*1*1变换为10*1*1
net = mx.sym.Convolution(data=net, name="final", kernel=(1,1), num_filter=10)
# 将10*1*1变换为10
net = mx.sym.Flatten(data=net, name="flatten")
net = mx.sym.SoftmaxOutput(data=net, name='softmax')

# 输出参数情况供参考
shape = {"data" : (batch_size, 1, 28, 28)}
mx.viz.print_summary(symbol=net, shape=shape)

# 由于训练数据多，这里采用了GPU，若读者没有GPU，可修改为CPU
module = mx.mod.Module(symbol=net, context=mx.cpu(0))

# 迭代器：测试数据
val_iter = mx.io.NDArrayIter(val_img, val_lbl, batch_size) 

# 手动循环40个epoch
for epoch in range(40):
    # 生成增强的图像，最佳方法是在另一进程执行，这里只是演示
    # 首先复制一份原始图像
    aug_img = train_img.copy()
    # 修改其中的每幅图像
    for i in range(aug_img.shape[0]):
        # 有50%概率做左右翻转
        if np.random.random() < 0.5:
            # aug_img[i][0]为第i号样本的0号通道，灰度图像只有0号通道
            # fliplr()用于左右翻转
            aug_img[i][0] = np.fliplr(aug_img[i][0])
            
        # 左右移动最多2个像素，注意randint(a,b)的范围为a到b-1
        amt = np.random.randint(0, 3)
        if amt > 0: # 如果需要移动…
            if np.random.random() < 0.5: # 左移动还是右移动？
                # pad()用于加上外衬，因移动后减少的区域需补零
                # 然后用[:]取所要的部分
                aug_img[i][0] = np.pad(aug_img[i][0],((0,0),(amt,0)), mode='constant')[:, :-amt]
            else:
                aug_img[i][0] = np.pad(aug_img[i][0],((0,0),(0,amt)), mode='constant')[:, amt:]
        
        # 上下移动最多2个像素
        amt = np.random.randint(0, 3)
        if amt > 0:
            if np.random.random() < 0.5:
                aug_img[i][0] = np.pad(aug_img[i][0],((amt,0),(0,0)), mode='constant')[:-amt, :]
            else:
                aug_img[i][0] = np.pad(aug_img[i][0],((0,amt),(0,0)), mode='constant')[amt:, :]
        
        # 随机清零最大5*5的区域
        x_size = np.random.randint(1, 6)
        y_size = np.random.randint(1, 6)
        x_begin = np.random.randint(0, 28-x_size+1)
        y_begin = np.random.randint(0, 28-y_size+1)
        aug_img[i][0][x_begin:x_begin+x_size, y_begin:y_begin+y_size] = 0
        
    # 每个epoch重设训练数据
    train_iter = mx.io.NDArrayIter(aug_img, train_lbl, batch_size, shuffle=True)
    # 每个epoch降低学习速率
    lr = 0.06 * pow(0.95, epoch)
    # 输出当前epoch信息
    print("epoch " + str(epoch) + ", learning rate = " + str(lr))
    # 训练
    module.fit(
        train_iter,
        eval_data=val_iter,
        optimizer = 'sgd', 
        optimizer_params = {'learning_rate' : lr},
        num_epoch = 1, # 每次训练1个epoch
        batch_end_callback = mx.callback.Speedometer(batch_size, 60000/batch_size)
    )

C:\ProgramData\Miniconda3\lib\site-packages\ipykernel_launcher.py:19: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
C:\ProgramData\Miniconda3\lib\site-packages\ipykernel_launcher.py:22: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead


________________________________________________________________________________________________________________________
Layer (type)                                        Output Shape            Param #     Previous Layer                  
data(null)                                          1x28x28                 0                                           
________________________________________________________________________________________________________________________
conv11(Convolution)                                 32x28x28                320         data                            
________________________________________________________________________________________________________________________
bn11(BatchNorm)                                     32x28x28                64          conv11                          
________________________________________________________________________________________________________________________
act11(Activation)               

INFO:root:Epoch[0] Train-accuracy=0.805233
INFO:root:Epoch[0] Time cost=368.177
INFO:root:Epoch[0] Validation-accuracy=0.864716
C:\ProgramData\Miniconda3\lib\site-packages\mxnet-1.4.0-py3.6.egg\mxnet\module\base_module.py:503: UserWarning: Parameters already initialized and force_init=False. init_params call ignored.
  allow_missing=allow_missing, force_init=force_init)


epoch 1, learning rate = 0.056999999999999995


INFO:root:Epoch[0] Train-accuracy=0.873267
INFO:root:Epoch[0] Time cost=365.364
INFO:root:Epoch[0] Validation-accuracy=0.886482


...

INFO:root:Epoch[0] Train-accuracy=0.950383
INFO:root:Epoch[0] Time cost=346.874
INFO:root:Epoch[0] Validation-accuracy=0.947584


epoch 38, learning rate = 0.00854374448078248


INFO:root:Epoch[0] Train-accuracy=0.951083
INFO:root:Epoch[0] Time cost=348.568
INFO:root:Epoch[0] Validation-accuracy=0.947784


epoch 39, learning rate = 0.008116557256743356


INFO:root:Epoch[0] Train-accuracy=0.950650
INFO:root:Epoch[0] Time cost=348.840
INFO:root:Epoch[0] Validation-accuracy=0.945487


In [6]:
#-*-coding:utf-8-*-
#import sys
#reload(sys)
#sys.setdefaultencoding('utf-8') # 进一步要求 python 使用 UTF-8

import numpy as np
import logging
import mxnet as mx

logging.getLogger().setLevel(logging.DEBUG)

batch_size = 50 # 批大小

# 用于辅助定义网络，这是深度卷积网络中很常用的"Conv-BN-Act"模块
def CBA(src, suffix, num_filter, kernel, pad, stride=(1,1)):
    conv = mx.sym.Convolution(src, name="conv"+suffix, kernel=(kernel,kernel), pad=(pad,pad), num_filter=num_filter, stride=stride)
    bn = mx.sym.BatchNorm(conv, name="bn"+suffix, fix_gamma=False)
    act = mx.sym.Activation(bn, name="act"+suffix, act_type="relu")
    return act

# 全卷积架构，由带步长的卷积实现缩小
net = mx.symbol.Variable('data')
# 将3*28*28变换为96*28*28
net = CBA(net, "1", 96, 3, 1)
# 将96*28*28变换为96*28*28
net = CBA(net, "2", 96, 3, 1)
# 将96*28*28变换为96*14*14
net = CBA(net, "3", 96, 3, 1, (2,2))
# 将96*14*14变换为192*14*14
net = CBA(net, "4", 192, 3, 1)
# 将192*14*14变换为192*14*14
net = CBA(net, "5", 192, 3, 1)
# 将192*14*14变换为192*7*7
net = CBA(net, "6", 192, 3, 1, (2,2))
# 将192*7*7变换为192*5*5
net = CBA(net, "7", 192, 3, 0)
# 将192*5*5变换为192*5*5
net = CBA(net, "8", 192, 1, 0)
# 将192*5*5变换为10*5*5
net = CBA(net, "9", 10, 1, 0)
# 将10*5*5变换为10*1*1
net = mx.sym.Pooling(net, name="pool", global_pool=True, pool_type="avg", kernel=(1,1))
# 将10*1*1变换为10
net = mx.sym.Flatten(net, name="flatten")
net = mx.sym.SoftmaxOutput(net, name='softmax')
 
# 输出参数情况供参考
shape = {"data" : (batch_size, 3, 28, 28)}
mx.viz.print_summary(symbol=net, shape=shape)

# 由于训练数据多，这里采用了GPU，若读者没有GPU，可修改为CPU
module = mx.mod.Module(symbol=net, context=mx.cpu(0))

# 迭代器，训练数据：
train_iter = mx.io.ImageRecordIter(
            path_imgrec = "train.rec",
            data_shape  = (3,28,28), # 图像通道和尺寸
            batch_size  = batch_size,
            shuffle = True, # 开启随机次序
            rand_crop   = True, # 开启随机裁剪
            rand_mirror = True, # 开启随机镜像
            random_h = 10, # 随机色相
            random_s = 20, # 随机饱和度
            random_l = 25, # 随机亮度
            max_random_scale = 1.20, # 随机放大
            min_random_scale = 0.88, # 随机缩小 
            max_rotate_angle = 20, # 随机旋转
            max_aspect_ratio = 0.15, # 随机长宽比例
            max_shear_ratio = 0.10, # 随机倾斜比例
            fill_value = 0, # 四周填充黑色 
) 
# 测试数据，关闭数据增强： 
val_iter = mx.io.ImageRecordIter(
            path_imgrec = "test.rec",
            data_shape  = (3,28,28),
            batch_size  = batch_size,
            shuffle = False,
            rand_crop   = False,
            rand_mirror = False,
) 
# 训练
module.fit(
    train_iter,
    eval_data=val_iter,
    initializer = mx.init.MSRAPrelu(slope=0.0), # 采用MSRAPrelu初始化
    optimizer = 'sgd', 
# 采用0.5的初始学习速率，并在每50000个样本后将学习速率缩减为之前的0.98倍
    optimizer_params = {'learning_rate' : 0.5, 'lr_scheduler' : mx.lr_scheduler.FactorScheduler(step=50000/batch_size, factor=0.98)},
    num_epoch = 200,
    batch_end_callback = mx.callback.Speedometer(batch_size, 50000/batch_size)
)

________________________________________________________________________________________________________________________
Layer (type)                                        Output Shape            Param #     Previous Layer                  
data(null)                                          3x28x28                 0                                           
________________________________________________________________________________________________________________________
conv1(Convolution)                                  96x28x28                2688        data                            
________________________________________________________________________________________________________________________
bn1(BatchNorm)                                      96x28x28                192         conv1                           
________________________________________________________________________________________________________________________
act1(Activation)                

INFO:root:Epoch[0] Train-accuracy=0.442480
INFO:root:Epoch[0] Time cost=1353.606
INFO:root:Epoch[0] Validation-accuracy=0.504100
INFO:root:Update[1001]: Change learning rate to 4.90000e-01
INFO:root:Epoch[1] Train-accuracy=0.614320
INFO:root:Epoch[1] Time cost=1356.172
INFO:root:Epoch[1] Validation-accuracy=0.679100
INFO:root:Update[2001]: Change learning rate to 4.80200e-01
INFO:root:Epoch[2] Train-accuracy=0.684360
INFO:root:Epoch[2] Time cost=1434.675
INFO:root:Epoch[2] Validation-accuracy=0.689900
INFO:root:Update[3001]: Change learning rate to 4.70596e-01
INFO:root:Epoch[3] Train-accuracy=0.723740
INFO:root:Epoch[3] Time cost=1361.564
INFO:root:Epoch[3] Validation-accuracy=0.761100
INFO:root:Update[4001]: Change learning rate to 4.61184e-01
INFO:root:Epoch[4] Train-accuracy=0.747100
INFO:root:Epoch[4] Time cost=1353.557
INFO:root:Epoch[4] Validation-accuracy=0.750100
INFO:root:Update[5001]: Change learning rate to 4.51960e-01
INFO:root:Epoch[5] Train-accuracy=0.767460
INFO:root:Ep

In [1]:
import mxnet as mx

# 每层的卷积神经元个数，AlphaGo Zero使用256个，我们为加快训练使用128个
n_filter = 128

net = mx.symbol.Variable('data')
# 预处理
net = mx.sym.Convolution(net, name='convPRE', kernel=(3, 3), pad=(1, 1), num_filter = n_filter)

# 残差结构
for i in range(0, 6):
    identity = net # 保存之前的输出
    net = mx.sym.BatchNorm(net, name='bnA'+str(i), fix_gamma=False)
    net = mx.sym.Activation(net, name='actA'+str(i), act_type='relu')
    net = mx.sym.Convolution(net, name='convA'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter)
    net = mx.sym.BatchNorm(net, name='bnB'+str(i), fix_gamma=False)
    net = mx.sym.Activation(net, name='actB'+str(i), act_type='relu')
    net = mx.sym.Convolution(net, name='convB'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter)
    net = net + identity # 直接加上之前的输出，即为残差结构

# 收尾工作
net = mx.sym.BatchNorm(net, name='bnFINAL', fix_gamma=False)
net = mx.sym.Activation(net, name='actFINAL', act_type='relu')
# 合并为1个通道
net = mx.sym.Convolution(net, name='convFINAL', kernel=(1, 1), num_filter=1)
net = mx.sym.Flatten(net)
# 输出为361个概率
net = mx.sym.SoftmaxOutput(net, name='softmax')

# 检查参数量
shape = {"data" : (32, 8, 19, 19)}
mx.viz.print_summary(symbol=net, shape=shape)

C:\ProgramData\Miniconda3\lib\site-packages\h5py\__init__.py:72: UserWarning: h5py is running against HDF5 1.10.2 when it was built against 1.10.3, this may cause problems
  '{0}.{1}.{2}'.format(*version.hdf5_built_version_tuple)


________________________________________________________________________________________________________________________
Layer (type)                                        Output Shape            Param #     Previous Layer                  
data(null)                                          8x19x19                 0                                           
________________________________________________________________________________________________________________________
convPRE(Convolution)                                128x19x19               9344        data                            
________________________________________________________________________________________________________________________
bnA0(BatchNorm)                                     128x19x19               256         convPRE                         
________________________________________________________________________________________________________________________
actA0(Activation)               

In [ ]:
import numpy as np
import logging
import mxnet as mx

logging.getLogger().setLevel(logging.DEBUG)

batch_size = 100 # 在此增大了批大小，因为ResNet的训练较慢

# 残差模块
def ResBlock(net, suffix, n_block, n_filter, stride=(1,1)):
    for i in range(0, n_block):
        if i == 0: # 注意第1个残差层的定义不同，读者可观察结构图思考原因
            net = mx.sym.BatchNorm(net, name='bn'+suffix+'_a'+str(i), fix_gamma=False)
            net = mx.sym.Activation(net, name='act'+suffix+'_a'+str(i), act_type='relu')
            # 对于第1个残差层，旁路从此开始
            pathway = mx.sym.Convolution(net, name="adj"+suffix, kernel=(1,1), stride=stride, num_filter=n_filter)
            # 回到主路
            net = mx.sym.Convolution(net, name='conv'+suffix+'_a'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter, stride=stride)
            net = mx.sym.BatchNorm(net, name='bn'+suffix+'_b'+str(i), fix_gamma=False)
            net = mx.sym.Activation(net, name='act'+suffix+'_b'+str(i), act_type='relu')
            net = mx.sym.Convolution(net, name='conv'+suffix+'_b'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter)
            net = net + pathway # 加上旁路，即为残差结构
        else:
            pathway = net # 对于后续残差层，旁路从此开始
            net = mx.sym.BatchNorm(net, name='bn'+suffix+'_a'+str(i), fix_gamma=False)
            net = mx.sym.Activation(net, name='act'+suffix+'_a'+str(i), act_type='relu')
            net = mx.sym.Convolution(net, name='conv'+suffix+'_a'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter)
            net = mx.sym.BatchNorm(net, name='bn'+suffix+'_b'+str(i), fix_gamma=False)
            net = mx.sym.Activation(net, name='act'+suffix+'_b'+str(i), act_type='relu')
            net = mx.sym.Convolution(net, name='conv'+suffix+'_b'+str(i), kernel=(3, 3), pad=(1, 1), num_filter = n_filter)
            net = net + pathway # 加上旁路，即为残差结构
    return net

net = mx.symbol.Variable('data')
# 为数据加上BN层可有一定的预处理效果
net = mx.sym.BatchNorm(net, name='bnPRE', fix_gamma=False)
# 将3*32*32变化为32*32*32
net = mx.sym.Convolution(net, name="convPRE", kernel=(3,3), pad=(1,1), num_filter=32)
# 将32*32*32变化为64*32*32
net = ResBlock(net, "1", 3, 64)
# 将64*32*32变化为64*16*16
net = ResBlock(net, "2", 3, 64, (2,2))
# 将64*16*16变化为128*8*8
net = ResBlock(net, "3", 3, 128, (2,2))
# 因为此前的最终层是卷积，因此再做BN和Act处理
net = mx.sym.BatchNorm(net, name='bnFINAL', fix_gamma=False)
net = mx.sym.Activation(net, name='actFINAL', act_type='relu')
# 将128*8*8变化为128*1*1
net = mx.sym.Pooling(net, name="pool", global_pool=True, pool_type="avg", kernel=(1,1))
# 将128*1*1变换为128
net = mx.sym.Flatten(net, name="flatten")
# 将128变换为10
net = mx.sym.FullyConnected(net, num_hidden=10, name='fc')
net = mx.sym.SoftmaxOutput(net, name='softmax')

# 输出参数情况供参考
shape = {"data" : (batch_size, 3, 28, 28)}
mx.viz.print_summary(symbol=net, shape=shape)

module = mx.mod.Module(symbol=net, context=mx.cpu(0)) # 网络模组

# 数据迭代器
train_iter = mx.io.ImageRecordIter(
            path_imgrec = "train.rec",
            data_shape  = (3,28,28), # 图像通道和尺寸
            batch_size  = batch_size,
            shuffle = True, # 开启随机次序
            rand_crop   = True, # 开启随机裁剪
            rand_mirror = True, # 开启随机镜像
            random_h = 10, # 随机色相
            random_s = 20, # 随机饱和度
            random_l = 25, # 随机亮度
            max_random_scale = 1.20, # 随机放大
            min_random_scale = 0.88, # 随机缩小 
            max_rotate_angle = 20, # 随机旋转
            max_aspect_ratio = 0.15, # 随机长宽比例
            max_shear_ratio = 0.10, # 随机倾斜比例
            fill_value = 0, # 四周填充黑色
) 
val_iter = mx.io.ImageRecordIter(
            path_imgrec = "test.rec",
            data_shape  = (3,28,28),
            batch_size  = batch_size,
            shuffle = False,
            rand_crop   = False,
            rand_mirror = False,
)

# 训练
module.fit(
    train_iter,
    eval_data=val_iter,
    initializer = mx.init.MSRAPrelu(slope=0.0), # 采用MSRAPrelu初始化
    optimizer = 'sgd', 
    # 采用0.5的初始学习速率，并在每50000个样本后将学习速率缩减为之前的0.984倍
    optimizer_params = {'learning_rate' : 0.5, 'lr_scheduler' : mx.lr_scheduler.FactorScheduler(step=50000/batch_size, factor=0.984)},
    num_epoch = 200,
    batch_end_callback = mx.callback.Speedometer(batch_size, 50000/batch_size)
)

________________________________________________________________________________________________________________________
Layer (type)                                        Output Shape            Param #     Previous Layer                  
data(null)                                          3x28x28                 0                                           
________________________________________________________________________________________________________________________
bnPRE(BatchNorm)                                    3x28x28                 6           data                            
________________________________________________________________________________________________________________________
convPRE(Convolution)                                32x28x28                896         bnPRE                           
________________________________________________________________________________________________________________________
bn1_a0(BatchNorm)               

INFO:root:Epoch[0] Train-accuracy=0.425700
INFO:root:Epoch[0] Time cost=2139.974
INFO:root:Epoch[0] Validation-accuracy=0.530700
INFO:root:Update[501]: Change learning rate to 4.92000e-01
INFO:root:Epoch[1] Train-accuracy=0.611860
INFO:root:Epoch[1] Time cost=2728.544
INFO:root:Epoch[1] Validation-accuracy=0.593200
INFO:root:Update[1001]: Change learning rate to 4.84128e-01
INFO:root:Epoch[2] Train-accuracy=0.680880
INFO:root:Epoch[2] Time cost=2681.169
INFO:root:Epoch[2] Validation-accuracy=0.691000
INFO:root:Update[1501]: Change learning rate to 4.76382e-01
INFO:root:Epoch[3] Train-accuracy=0.720900
INFO:root:Epoch[3] Time cost=2590.392
INFO:root:Epoch[3] Validation-accuracy=0.764900
INFO:root:Update[2001]: Change learning rate to 4.68760e-01
INFO:root:Epoch[4] Train-accuracy=0.747780
INFO:root:Epoch[4] Time cost=2568.334
INFO:root:Epoch[4] Validation-accuracy=0.747200
INFO:root:Update[2501]: Change learning rate to 4.61260e-01
INFO:root:Epoch[5] Train-accuracy=0.768940
INFO:root:Epo